# Análisis de Housing con pandas

Este notebook reproduce en **pandas** las operaciones solicitadas originalmente para Spark:

1. Configuración y validación del entorno.
2. Importación del archivo `kc_house_data.csv`.
3. Listado completo de columnas ordenado por `zipcode`.
4. Identificación del `zipcode` con mayor número de casas.
5. Cálculo del precio promedio y tamaño habitable promedio en m² para ese `zipcode`.
6. Agrupamiento por `zipcode`, número de habitaciones y baños, calculando el precio promedio.


## 1. Configuración de la plataforma pandas

In [1]:
# Importación de librerías
from pathlib import Path
import sys
import pandas as pd
import numpy as np

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.11.15
pandas: 3.0.3
NumPy: 2.4.6


## 2. Importación de los datos de Housing

In [2]:
# El CSV debe estar en la misma carpeta que este notebook.
# También se incluye una ruta alternativa para ejecutarlo desde /mnt/data.

posibles_rutas = [
    Path("../../../Archivos-Analisis/files-tarea-m40/kc_house_data.csv")
]

ruta_csv = next((ruta for ruta in posibles_rutas if ruta.exists()), None)

if ruta_csv is None:
    raise FileNotFoundError(
        "No se encontró 'kc_house_data.csv'. "
        "Coloca el CSV en la misma carpeta que el notebook."
    )

housing = pd.read_csv(ruta_csv)

print(f"Archivo cargado: {ruta_csv.resolve()}")
print(f"Filas: {housing.shape[0]:,}")
print(f"Columnas: {housing.shape[1]}")

Archivo cargado: C:\Users\RyanHz\Documents\EBAC\VS\Archivos-Analisis\files-tarea-m40\kc_house_data.csv
Filas: 21,613
Columnas: 21


In [3]:
# Vista previa del DataFrame
housing.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [4]:
# Nombres y tipos de datos de las columnas
housing.info()

<class 'pandas.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  str    
 2   price          21613 non-null  float64
 3   bedrooms       21613 non-null  int64  
 4   bathrooms      21613 non-null  float64
 5   sqft_living    21613 non-null  int64  
 6   sqft_lot       21613 non-null  int64  
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21613 non-null  int64  
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat            21613 non-null  float64
 18  long           21

In [5]:
# Validación de columnas requeridas
columnas_requeridas = {
    "zipcode", "price", "bedrooms", "bathrooms", "sqft_living"
}

columnas_faltantes = columnas_requeridas.difference(housing.columns)

if columnas_faltantes:
    raise ValueError(
        f"Faltan columnas requeridas en el CSV: {sorted(columnas_faltantes)}"
    )

print("Todas las columnas necesarias están disponibles.")

Todas las columnas necesarias están disponibles.


## 3. Selección de datos con filtros simples

### 3.1 Listado completo de columnas ordenado por `zipcode`

Se ordenan todas las filas de forma ascendente por código postal. Como criterio secundario se utiliza `id`, para mantener un orden estable dentro de cada `zipcode`.


In [6]:
housing_ordenado = (
    housing
    .sort_values(by=["zipcode", "id"], ascending=[True, True])
    .reset_index(drop=True)
)

housing_ordenado

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,128500260,20140508T000000,262000.0,4,2.5,2020,6236,2.0,0,0,3,7,2020,0,2002,0,98001,47.2796,-122.247,1940,5076
1,221049191,20150428T000000,329500.0,3,2.5,2120,22482,1.0,0,0,5,7,1360,760,1979,0,98001,47.3410,-122.265,2330,16016
2,255580190,20140915T000000,302000.0,4,2.5,1740,7895,2.0,0,0,3,7,1740,0,1999,0,98001,47.3401,-122.282,1720,6813
3,302000065,20150129T000000,184000.0,3,1.0,970,14850,1.0,0,0,3,7,970,0,1968,0,98001,47.3251,-122.268,1410,14850
4,302000375,20140814T000000,169100.0,3,2.0,1050,18304,1.0,0,0,4,7,1050,0,1953,0,98001,47.3206,-122.269,1690,15675
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,8127700820,20141211T000000,640000.0,3,2.0,1470,4640,1.5,0,0,4,7,1470,0,1926,0,98199,47.6398,-122.393,1700,5000
21609,8127700845,20150219T000000,375000.0,2,1.0,710,4618,1.0,0,1,3,5,710,0,1925,0,98199,47.6400,-122.394,1810,4988
21610,8941100095,20140923T000000,1112500.0,6,4.0,3600,6224,2.0,0,0,3,9,2610,990,1945,2006,98199,47.6531,-122.405,1430,6224
21611,9241900115,20150324T000000,1100000.0,4,3.0,3320,5760,2.0,0,0,3,9,2120,1200,1954,2007,98199,47.6474,-122.389,2400,6144


### 3.2 `zipcode` con mayor número de casas

Primero se cuenta cuántas viviendas existen en cada código postal.


In [7]:
casas_por_zipcode = (
    housing
    .groupby("zipcode", as_index=False)
    .size()
    .rename(columns={"size": "numero_casas"})
    .sort_values(
        by=["numero_casas", "zipcode"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

casas_por_zipcode.head(10)

,zipcode,numero_casas
0,98103,602
1,98038,590
2,98115,583
3,98052,574
4,98117,553
5,98042,548
6,98034,545
7,98118,508
8,98023,499
9,98006,498


In [8]:
zipcode_mas_casas = int(casas_por_zipcode.loc[0, "zipcode"])
numero_casas = int(casas_por_zipcode.loc[0, "numero_casas"])

print(f"Zipcode con mayor número de casas: {zipcode_mas_casas}")
print(f"Número de casas: {numero_casas:,}")

Zipcode con mayor número de casas: 98103
Número de casas: 602


### Precio promedio y tamaño promedio en m²

La columna `sqft_living` está expresada en pies cuadrados. Se convierte a metros cuadrados con:

`1 pie² = 0.092903 m²`


In [9]:
PIE_CUADRADO_A_M2 = 0.092903

housing = housing.copy()
housing["living_m2"] = housing["sqft_living"] * PIE_CUADRADO_A_M2

casas_zipcode_mayor = housing.loc[
    housing["zipcode"] == zipcode_mas_casas
].copy()

resumen_zipcode_mayor = pd.DataFrame({
    "zipcode": [zipcode_mas_casas],
    "numero_casas": [len(casas_zipcode_mayor)],
    "precio_promedio": [casas_zipcode_mayor["price"].mean()],
    "tamano_promedio_m2": [casas_zipcode_mayor["living_m2"].mean()]
})

resumen_zipcode_mayor.style.format({
    "numero_casas": "{:,.0f}",
    "precio_promedio": "${:,.2f}",
    "tamano_promedio_m2": "{:,.2f} m²"
})

,zipcode,numero_casas,precio_promedio,tamano_promedio_m2
0,98103,602,"$584,919.21",153.37 m²


In [10]:
# Resultado también en texto
precio_promedio = casas_zipcode_mayor["price"].mean()
tamano_promedio_m2 = casas_zipcode_mayor["living_m2"].mean()

print(f"Zipcode: {zipcode_mas_casas}")
print(f"Casas: {len(casas_zipcode_mayor):,}")
print(f"Precio promedio: ${precio_promedio:,.2f}")
print(f"Tamaño habitable promedio: {tamano_promedio_m2:,.2f} m²")

Zipcode: 98103
Casas: 602
Precio promedio: $584,919.21
Tamaño habitable promedio: 153.37 m²


## 4. Agrupamiento por habitaciones, baños y `zipcode`

Se agrupan las viviendas usando:

- `zipcode`
- `bedrooms`
- `bathrooms`

Para cada combinación se calcula:

- Precio promedio.
- Número de casas, como dato adicional para interpretar mejor el promedio.


In [11]:
precio_promedio_por_grupo = (
    housing
    .groupby(
        ["zipcode", "bedrooms", "bathrooms"],
        as_index=False,
        dropna=False
    )
    .agg(
        precio_promedio=("price", "mean"),
        numero_casas=("id", "count")
    )
    .sort_values(
        by=["zipcode", "bedrooms", "bathrooms"],
        ascending=[True, True, True]
    )
    .reset_index(drop=True)
)

precio_promedio_por_grupo.head(20).style.format({
    "precio_promedio": "${:,.2f}",
    "numero_casas": "{:,.0f}"
})

,zipcode,bedrooms,bathrooms,precio_promedio,numero_casas
0,98001,0,0.000000,"$139,950.00",1
1,98001,1,1.000000,"$166,000.00",1
2,98001,1,2.000000,"$171,000.00",2
3,98001,2,1.000000,"$197,428.57",14
4,98001,2,1.500000,"$350,000.00",1
5,98001,2,1.750000,"$246,112.50",4
6,98001,2,2.500000,"$214,100.00",1
7,98001,2,2.750000,"$239,475.00",2
8,98001,3,0.750000,"$363,000.00",1
9,98001,3,1.000000,"$205,182.81",42


### Consulta opcional para un `zipcode` específico

In [12]:
# Cambia este valor para consultar otro código postal.
zipcode_consulta = zipcode_mas_casas

resultado_zipcode = precio_promedio_por_grupo.loc[
    precio_promedio_por_grupo["zipcode"] == zipcode_consulta
].reset_index(drop=True)

resultado_zipcode.style.format({
    "precio_promedio": "${:,.2f}",
    "numero_casas": "{:,.0f}"
})

,zipcode,bedrooms,bathrooms,precio_promedio,numero_casas
0,98103,1,1.000000,"$395,145.45",11
1,98103,1,2.500000,"$680,000.00",1
2,98103,2,1.000000,"$486,496.36",88
3,98103,2,1.500000,"$422,898.21",28
4,98103,2,1.750000,"$522,380.77",13
5,98103,2,2.000000,"$598,893.11",9
6,98103,2,2.250000,"$496,966.67",9
7,98103,2,2.500000,"$436,995.83",12
8,98103,2,2.750000,"$525,000.00",1
9,98103,2,3.000000,"$545,000.00",2


## 5. Exportación opcional de resultados

In [13]:
# Descomenta estas líneas para guardar los resultados en archivos CSV.

# housing_ordenado.to_csv(
#     "housing_ordenado_por_zipcode.csv",
#     index=False
# )

# resumen_zipcode_mayor.to_csv(
#     "resumen_zipcode_mayor_numero_casas.csv",
#     index=False
# )

# precio_promedio_por_grupo.to_csv(
#     "precio_promedio_por_zipcode_habitaciones_banos.csv",
#     index=False
# )

print("Análisis completado correctamente.")

Análisis completado correctamente.
